# Day 3: Team-Perspective Dataset

Today we convert match-level data into team-level rows so we can calculate form and strength features later.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
matches = pd.read_csv("../data/processed/matches_with_results.csv")
matches["date"] = pd.to_datetime(matches["date"])
matches.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,draw
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,home_win
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,home_win
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,draw
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,home_win


In [6]:
home_rows = pd.DataFrame({
    "date": matches["date"],
    "team": matches["home_team"],
    "opponent": matches["away_team"],
    "goals_for": matches["home_score"],
    "goals_against": matches["away_score"],
    "tournament": matches["tournament"],
    "city": matches["city"],
    "country": matches["country"],
    "neutral": matches["neutral"],
    "is_home": True
})

home_rows.head()

,date,team,opponent,goals_for,goals_against,tournament,city,country,neutral,is_home
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,True
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,True
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,True
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,True
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,True


In [7]:
away_rows = pd.DataFrame({
    "date": matches["date"],
    "team": matches["away_team"],
    "opponent": matches["home_team"],
    "goals_for": matches["away_score"],
    "goals_against": matches["home_score"],
    "tournament": matches["tournament"],
    "city": matches["city"],
    "country": matches["country"],
    "neutral": matches["neutral"],
    "is_home": False
})

away_rows.head()

,date,team,opponent,goals_for,goals_against,tournament,city,country,neutral,is_home
0,1872-11-30,England,Scotland,0.0,0.0,Friendly,Glasgow,Scotland,False,False
1,1873-03-08,Scotland,England,2.0,4.0,Friendly,London,England,False,False
2,1874-03-07,England,Scotland,1.0,2.0,Friendly,Glasgow,Scotland,False,False
3,1875-03-06,Scotland,England,2.0,2.0,Friendly,London,England,False,False
4,1876-03-04,England,Scotland,0.0,3.0,Friendly,Glasgow,Scotland,False,False


In [8]:
team_matches = pd.concat([home_rows, away_rows], ignore_index=True)
team_matches.head()

,date,team,opponent,goals_for,goals_against,tournament,city,country,neutral,is_home
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,True
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,True
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,True
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,True
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,True


In [9]:
matches.shape, team_matches.shape

((49450, 10), (98900, 10))

In [10]:
def get_team_result(row):
    if row["goals_for"] > row["goals_against"]:
        return "win"
    elif row["goals_for"] < row["goals_against"]:
        return "loss"
    else:
        return "draw"

team_matches["result"] = team_matches.apply(get_team_result, axis=1)

In [11]:
team_matches.head()

,date,team,opponent,goals_for,goals_against,tournament,city,country,neutral,is_home,result
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,True,draw
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,True,win
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,True,win
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,True,draw
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,True,win


In [12]:
def get_points(result):
    if result == "win":
        return 3
    elif result == "draw":
        return 1
    else:
        return 0

team_matches["points"] = team_matches["result"].apply(get_points)

In [14]:
team_matches.head(10)

,date,team,opponent,goals_for,goals_against,tournament,city,country,neutral,is_home,result,points
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,True,draw,1
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,True,win,3
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,True,win,3
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,True,draw,1
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,True,win,3
5,1876-03-25,Scotland,Wales,4.0,0.0,Friendly,Glasgow,Scotland,False,True,win,3
6,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False,True,loss,0
7,1877-03-05,Wales,Scotland,0.0,2.0,Friendly,Wrexham,Wales,False,True,loss,0
8,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False,True,win,3
9,1878-03-23,Scotland,Wales,9.0,0.0,Friendly,Glasgow,Scotland,False,True,win,3


In [15]:
team_matches["goal_difference"] = team_matches["goals_for"] - team_matches["goals_against"]

In [17]:
team_matches[["team", "opponent", "goals_for", "goals_against", "goal_difference"]].head()

,team,opponent,goals_for,goals_against,goal_difference
0,Scotland,England,0.0,0.0,0.0
1,England,Scotland,4.0,2.0,2.0
2,Scotland,England,2.0,1.0,1.0
3,England,Scotland,2.0,2.0,0.0
4,Scotland,England,3.0,0.0,3.0


In [18]:
team_matches = team_matches.sort_values(["team", "date"]).reset_index(drop=True)

In [19]:
team_matches[team_matches["team"] == "Brazil"].tail(10)

,date,team,opponent,goals_for,goals_against,tournament,city,country,neutral,is_home,result,points,goal_difference
11151,2025-10-14,Brazil,Japan,2.0,3.0,Kirin Cup,Tokyo,Japan,False,False,loss,0,-1.0
11152,2025-11-15,Brazil,Senegal,2.0,0.0,Friendly,London,England,True,True,win,3,2.0
11153,2025-11-18,Brazil,Tunisia,1.0,1.0,Friendly,Lille,France,True,True,draw,1,0.0
11154,2026-03-26,Brazil,France,1.0,2.0,Friendly,Foxborough,United States,True,True,loss,0,-1.0
11155,2026-03-31,Brazil,Croatia,3.0,1.0,Friendly,Orlando,United States,True,True,win,3,2.0
11156,2026-05-31,Brazil,Panama,6.0,2.0,Friendly,Rio de Janeiro,Brazil,False,True,win,3,4.0
11157,2026-06-06,Brazil,Egypt,2.0,1.0,Friendly,Cleveland,United States,True,True,win,3,1.0
11158,2026-06-13,Brazil,Morocco,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,True,draw,1,NaN
11159,2026-06-19,Brazil,Haiti,NaN,NaN,FIFA World Cup,Philadelphia,United States,True,True,draw,1,NaN
11160,2026-06-24,Brazil,Scotland,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,False,draw,1,NaN


In [20]:
team_matches.shape
team_matches["result"].value_counts()
team_matches["result"].value_counts()

result
loss    38152
win     38152
draw    22596
Name: count, dtype: int64

In [21]:
team_matches.isna().sum()

date                 0
team                 0
opponent             0
goals_for          144
goals_against      144
tournament           0
city                 0
country              0
neutral              0
is_home              0
result               0
points               0
goal_difference    144
dtype: int64

In [22]:
team_matches.to_csv("../data/processed/team_matches.csv", index=False)